# 00 — Setup, Connectivity & Resilience  
### *Coding Alone* · ICSE 2027 SEIS pipeline

Prepares everything the pipeline needs **and installs the durability layer** used by every later notebook.

1. Authenticate this Colab session to Google Cloud (**ANON@institution**).
2. Connect to **BigQuery**; confirm the public **GH Archive** dataset is queryable.
3. Mount Google Drive; create the project + checkpoint folders.
4. Install the **resilience store**: completed-unit manifests, atomic file writes, a persistent BigQuery **job ledger** (so we reattach to server-side jobs after a disconnect), and a retry/backoff helper.
5. Install the cost-safety **`run_query`** helper (dry-run + GB cap + job persistence).
6. Run a tiny **live test query** to prove the chain end to end.

**Durability guarantee established here:** every unit of work is idempotent and checkpointed. If Colab disconnects, a rate limit trips, or a query times out, re-running a notebook **skips completed units and loses nothing**.


## 1. Configuration — edit this one cell
Paste your Cloud **project ID** (looks like `YOUR-GCP-PROJECT-ID-472301`).

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"     # <-- YOUR real project ID goes HERE, in these quotes
# ==========================================================================

# Panel window. Data only exists up to the latest COMPLETE month on GH Archive.
# Running mid-2026 => coverage is 2018-01 .. latest complete month (~202606).
WINDOW_START = "201801"       # YYYYMM
WINDOW_END   = "202606"       # YYYYMM  <-- bump to the latest COMPLETE month at run time

BQ_DATASET   = "seis"
BQ_LOCATION  = "US"           # GH Archive is US multi-region — keep US.
MAX_SCAN_GB  = 1500           # per-query hard cap (~$9.4). Guards against accidents.
DRIVE_BASE   = "/content/drive/MyDrive/ICSE_SEIS"

assert PROJECT_ID != "PASTE-YOUR-PROJECT-ID-HERE", "Set PROJECT_ID first."
print("Config loaded. PROJECT_ID =", PROJECT_ID, "| window", WINDOW_START, "->", WINDOW_END)

## 2. Authenticate & connect to BigQuery
Sign in as **ANON@institution**. Works for both a $300-trial project and a free **Sandbox** project.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print("BigQuery client ready for project:", client.project)


## 3. Mount Drive & create folders
Checkpoints, exports, and figures live on Drive so nothing dies with the runtime.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
for sub in ["", "/data", "/data/raw", "/data/derived", "/figures", "/tables", "/checkpoints"]:
    os.makedirs(DRIVE_BASE + sub, exist_ok=True)
print("Folders ready under", DRIVE_BASE)


## 4. Create the BigQuery dataset
Holds every derived table inside **your** project.

In [ ]:
ds = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
ds.location = BQ_LOCATION
ds = client.create_dataset(ds, exists_ok=True)
print("Dataset ready:", f"{PROJECT_ID}.{BQ_DATASET}")


## 5. Resilience store  ⟵ *the "never lose data" layer*
Defines, on Drive so it survives runtime death:

- `manifest_done(name)` / `manifest_mark(name, key)` — the set of completed units; loops skip what's done.
- `atomic_write_df(df, path)` / `atomic_read_df(path)` — crash-safe parquet checkpoints (temp-then-rename).
- a persistent **BigQuery job ledger** (`bq_jobs.json`) so `run_query` can **reattach** to a job that kept running server-side after a disconnect, instead of recomputing/re-billing it.
- `with_retries(...)` — exponential-backoff decorator (used by the GitHub-API notebook for rate limits).

In [ ]:
import json, time, functools
CKPT_DIR = f"{DRIVE_BASE}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def _atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)                      # atomic on same filesystem

def load_json(name, default):
    p = f"{CKPT_DIR}/{name}"
    if os.path.exists(p):
        with open(p) as f: return json.load(f)
    return default
def save_json(name, obj):
    _atomic_write_text(f"{CKPT_DIR}/{name}", json.dumps(obj, indent=2, default=str))

def manifest_done(name):            # -> set of completed unit keys
    return set(load_json(f"manifest_{name}.json", []))
def manifest_mark(name, key):
    s = manifest_done(name); s.add(str(key))
    save_json(f"manifest_{name}.json", sorted(s)); return s

def atomic_write_df(df, path):
    tmp = path + ".tmp"; df.to_parquet(tmp, index=False); os.replace(tmp, path)
def atomic_read_df(path):
    import pandas as pd
    return pd.read_parquet(path) if os.path.exists(path) else None

_job_records = load_json("bq_jobs.json", {})   # job_key -> {job_id, location}
def _save_job_records(): save_json("bq_jobs.json", _job_records)

def with_retries(max_tries=6, base=2.0, exc=(Exception,)):
    def deco(fn):
        @functools.wraps(fn)
        def wrap(*a, **k):
            for i in range(max_tries):
                try: return fn(*a, **k)
                except exc as e:
                    if i == max_tries - 1: raise
                    w = base ** i
                    print(f"  retry {i+1}/{max_tries} after: {e} -> sleep {w:.0f}s"); time.sleep(w)
        return wrap
    return deco

print("Resilience store ready at", CKPT_DIR)
print("Recovered BQ job keys:", list(_job_records))


## 6. `run_query` — cost guard **+** job persistence
Every BigQuery call goes through this. It:
- **reattaches** to a completed server-side job if this `job_key` already ran (no recompute, no re-bill),
- **dry-runs** first and prints GB + $ estimate; **aborts** above `MAX_SCAN_GB`,
- persists the **job_id to Drive before waiting**, so a disconnect is recoverable,
- sets `maximum_bytes_billed` so BigQuery hard-stops a runaway query.

In [ ]:
def run_query(sql, destination=None, write="WRITE_TRUNCATE", override=False,
              to_dataframe=False, label=None, job_key=None):
    tag = f"[{label}] " if label else ""
    # 0) Reattach to a previously-submitted job that finished server-side.
    if job_key and job_key in _job_records:
        rec = _job_records[job_key]
        try:
            prev = client.get_job(rec["job_id"], location=rec.get("location", BQ_LOCATION))
            if prev.state == "DONE" and prev.error_result is None:
                print(f"{tag}Reusing finished job {rec['job_id']} (no recompute).")
                r = prev.result()
                return r.to_dataframe() if to_dataframe else r
        except Exception as e:
            print(f"{tag}(could not reattach: {e}; resubmitting)")
    # 1) Dry-run cost check.
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f"{tag}Will scan ~{gb:,.1f} GB  (~${gb/1000*6.25:,.2f})")
    if gb > MAX_SCAN_GB and not override:
        raise RuntimeError(f"ABORT: {gb:,.1f} GB > MAX_SCAN_GB={MAX_SCAN_GB}. Pass override=True to force.")
    # 2) Submit; persist job_id BEFORE blocking on the result.
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(MAX_SCAN_GB*1e9*1.05))
    if destination:
        cfg.destination = f"{PROJECT_ID}.{BQ_DATASET}.{destination}"
        cfg.write_disposition = write
    job = client.query(sql, job_config=cfg)
    if job_key:
        _job_records[job_key] = {"job_id": job.job_id, "location": BQ_LOCATION}; _save_job_records()
    r = job.result()
    print(f"{tag}Done. Scanned {job.total_bytes_processed/1e9:,.1f} GB"
          + (f" -> {destination}" if destination else ""))
    return r.to_dataframe() if to_dataframe else r

print("run_query() ready (idempotent + recoverable).")


## 7. Live end-to-end test
One day of GH Archive (a few GB) to confirm auth + BigQuery + GH Archive all work.

In [ ]:
df_test = run_query("""
SELECT type, COUNT(*) AS n
FROM `githubarchive.day.20240101`
GROUP BY type ORDER BY n DESC LIMIT 10
""", to_dataframe=True, label="connectivity-test")
df_test


## Done ✅
Event types above = the full chain works, and the resilience layer is live.

Optional budget alert (belt-and-suspenders): Cloud Console → **Billing → Budgets & alerts** → create `$50` budget.

Next: **`01_sampling_frame.ipynb`** — resumable, idempotent build of the ~50k-repo frame.